# Condition B — Frozen CLIP + Prompted BLIP-2
Formula: `vi = α·ϕV(x̂i) + (1−α)·ϕT(ci)`, ‖vi‖ = 1

- CLIP embeddings computed fresh
- BLIP-2 captions generated with product-focused prompt
- Two alpha values: 0.7 and 0.5
- Full image and cropped variants
- **Enable GPU before running**

In [ ]:

!pip install git+https://github.com/openai/CLIP.git -q
!pip install transformers accelerate -q
!pip install faiss-cpu -q
print('Installs done.')

In [ ]:

import os, json, math, random, zipfile
import numpy as np
import pandas as pd
import torch
import clip
import faiss
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from IPython.display import FileLink

SEED = 2023006

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
print(f'Seed   : {SEED}')

In [ ]:

ROOT     = '/kaggle/input/datasets/dedeepyaavancha/deepfashion-in-shop-clothes-retrieval-benchmark/In-shop Clothes Retrieval Benchmark'
SAVE_DIR = '/kaggle/working/condition_b'
os.makedirs(SAVE_DIR, exist_ok=True)
print('ROOT exists:', os.path.exists(ROOT))

In [ ]:

eval_path = os.path.join(ROOT, 'Eval', 'list_eval_partition.txt')
df = pd.read_csv(
    eval_path, sep=r'\s+', skiprows=2, header=None,
    names=['image_name', 'item_id', 'split']
)

bbox_path = os.path.join(ROOT, 'Anno', 'list_bbox_inshop.txt')
df_bbox = pd.read_csv(
    bbox_path, sep=r'\s+', skiprows=2, header=None,
    names=['image_name', 'clothes_type', 'pose_type', 'x1', 'y1', 'x2', 'y2']
)

df = df.merge(df_bbox, on='image_name', how='left')
df['image_path'] = df['image_name'].apply(lambda x: os.path.join(ROOT, x))

df_query   = df[df['split'] == 'query'].reset_index(drop=True)
df_gallery = df[df['split'] == 'gallery'].reset_index(drop=True)

# item_id lists — built from df, no JSON needed
gallery_ids     = df_gallery['item_id'].tolist()
query_ids       = df_query['item_id'].tolist()
gallery_ids_arr = np.array(gallery_ids)

print(f'Total images : {len(df)}')
print(f'Gallery      : {len(df_gallery)}')
print(f'Query        : {len(df_query)}')
print(f'Unique items : {df["item_id"].nunique()}')
print(f'  Gallery unique: {df_gallery["item_id"].nunique()}')
print(f'  Query unique  : {df_query["item_id"].nunique()}')

In [ ]:

clip_model, preprocess = clip.load('ViT-B/32', device=device)
clip_model.eval()
print('CLIP loaded: ViT-B/32')

In [ ]:

class FashionDataset(Dataset):
    def __init__(self, df, preprocess, use_bbox=False):
        self.df        = df.reset_index(drop=True)
        self.preprocess = preprocess
        self.use_bbox  = use_bbox

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        if self.use_bbox:
            img = img.crop((row['x1'], row['y1'], row['x2'], row['y2']))
        return self.preprocess(img), row['item_id'], row['image_name']

def embed_split(df, use_bbox=False, batch_size=256):
    set_seed(SEED)
    dataset = FashionDataset(df, preprocess, use_bbox=use_bbox)
    loader  = DataLoader(dataset, batch_size=batch_size,
                         num_workers=2, pin_memory=True)
    all_embs, all_ids, all_names = [], [], []
    with torch.no_grad():
        for i, (imgs, item_ids, names) in enumerate(loader):
            embs = clip_model.encode_image(imgs.to(device))
            embs = embs / embs.norm(dim=-1, keepdim=True)
            all_embs.append(embs.cpu().numpy())
            all_ids.extend(item_ids)
            all_names.extend(names)
            if i % 10 == 0:
                print(f'  batch {i}/{len(loader)}')
    return np.vstack(all_embs), all_ids, all_names

print('Embedding gallery (full)...')
gallery_embs_full, _, _ = embed_split(df_gallery, use_bbox=False)
print('Embedding gallery (crop)...')
gallery_embs_crop, _, _ = embed_split(df_gallery, use_bbox=True)
print('Embedding query (full)...')
query_embs_full,   _, _ = embed_split(df_query,   use_bbox=False)
print('Embedding query (crop)...')
query_embs_crop,   _, _ = embed_split(df_query,   use_bbox=True)

# Save embeddings immediately
np.save(f'{SAVE_DIR}/gallery_embs_full.npy', gallery_embs_full)
np.save(f'{SAVE_DIR}/gallery_embs_crop.npy', gallery_embs_crop)
np.save(f'{SAVE_DIR}/query_embs_full.npy',   query_embs_full)
np.save(f'{SAVE_DIR}/query_embs_crop.npy',   query_embs_crop)

print(f'\nGallery full : {gallery_embs_full.shape}')
print(f'Gallery crop : {gallery_embs_crop.shape}')
print(f'Query full   : {query_embs_full.shape}')
print(f'Query crop   : {query_embs_crop.shape}')
print('Embeddings saved.')

In [ ]:

blip_processor = Blip2Processor.from_pretrained('Salesforce/blip2-opt-2.7b')
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    'Salesforce/blip2-opt-2.7b',
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to(device)
blip_model.eval()
print('BLIP-2 loaded.')

In [ ]:

#
# Prompt design:
#   - QA format works best with blip2-opt (instruction-tuned)
#   - Explicitly lists attributes from the PDF spec:
#     color, fit, material, style details
#   - Suppresses model/person/background description
#   - max_new_tokens=60 gives enough room for detail
#
CAPTION_PROMPT = (
    "Describe the clothing item only. "
    "Do not mention any person, model, or background. "
    "Do not use phrases like 'wearing' or 'a person'. "
    "Include garment type, colors, material, fit, sleeves, and patterns. "
    "Answer in one concise sentence."
)

def generate_captions(df, use_bbox=False, batch_size=32):
    set_seed(SEED)
    all_captions = []
    df   = df.reset_index(drop=True)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    for start in range(0, len(df), batch_size):
        batch  = df.iloc[start:start+batch_size]
        images = []
        for _, row in batch.iterrows():
            img = Image.open(row['image_path']).convert('RGB')
            if use_bbox:
                img = img.crop((row['x1'], row['y1'], row['x2'], row['y2']))
            images.append(img)

        prompts = [CAPTION_PROMPT] * len(images)
        inputs  = blip_processor(
            images=images, text=prompts,
            return_tensors='pt', padding=True
        ).to(device, dtype)

        with torch.no_grad():
            out = blip_model.generate(**inputs, max_new_tokens=60)
        captions = blip_processor.batch_decode(out, skip_special_tokens=True)
        # Strip echoed prompt prefix if any
        captions = [c.replace(CAPTION_PROMPT, '').strip() for c in captions]
        all_captions.extend(captions)

        if start % (batch_size * 10) == 0:
            print(f'  {start}/{len(df)} | "{captions[0]}"')

    return all_captions

print('Generating gallery captions (full)...')
gallery_captions_full = generate_captions(df_gallery, use_bbox=False)
print('\nGenerating gallery captions (crop)...')
gallery_captions_crop = generate_captions(df_gallery, use_bbox=True)
print('\nGenerating query captions (full)...')
query_captions_full   = generate_captions(df_query,   use_bbox=False)
print('\nGenerating query captions (crop)...')
query_captions_crop   = generate_captions(df_query,   use_bbox=True)

# Save captions immediately
for name, data in [
    ('gallery_captions_full', gallery_captions_full),
    ('gallery_captions_crop', gallery_captions_crop),
    ('query_captions_full',   query_captions_full),
    ('query_captions_crop',   query_captions_crop),
]:
    with open(f'{SAVE_DIR}/{name}.json', 'w') as f:
        json.dump(data, f)

print('\nAll captions saved.')
print(f'\nSample gallery full : "{gallery_captions_full[0]}"')
print(f'Sample gallery crop : "{gallery_captions_crop[0]}"')
print(f'Sample query  full  : "{query_captions_full[0]}"')
print(f'Sample query  crop  : "{query_captions_crop[0]}"')

In [ ]:

def encode_text(captions, batch_size=512):
    set_seed(SEED)
    all_embs = []
    for start in range(0, len(captions), batch_size):
        tokens = clip.tokenize(
            captions[start:start+batch_size], truncate=True
        ).to(device)
        with torch.no_grad():
            embs = clip_model.encode_text(tokens)
            embs = embs / embs.norm(dim=-1, keepdim=True)
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs)

def fuse(img_embs, txt_embs, alpha):
    """vi = α·ϕV(x̂) + (1-α)·ϕT(c),  ‖vi‖=1"""
    fused = alpha * img_embs + (1 - alpha) * txt_embs
    return fused / np.linalg.norm(fused, axis=1, keepdims=True)

print('Encoding gallery text (full)...')
gallery_txt_full = encode_text(gallery_captions_full)
print('Encoding gallery text (crop)...')
gallery_txt_crop = encode_text(gallery_captions_crop)
print('Encoding query text (full)...')
query_txt_full   = encode_text(query_captions_full)
print('Encoding query text (crop)...')
query_txt_crop   = encode_text(query_captions_crop)
print('All text embeddings done.')

In [ ]:

def compute_metrics(query_ids, indices, gallery_ids_arr, Ks=[5, 10, 15]):
    results = {}
    for K in Ks:
        recalls, ndcgs, aps = [], [], []
        for q_idx, q_id in enumerate(query_ids):
            retrieved = gallery_ids_arr[indices[q_idx, :K]]
            relevant  = (retrieved == q_id)
            recalls.append(1.0 if relevant.any() else 0.0)
            dcg  = sum(rel / math.log2(r + 2) for r, rel in enumerate(relevant))
            idcg = sum(1.0 / math.log2(r + 2)
                       for r in range(min(int(relevant.sum()), K)))
            ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
            hits, prec = 0, 0.0
            for r, rel in enumerate(relevant):
                if rel:
                    hits += 1
                    prec += hits / (r + 1)
            aps.append(prec / min(int(relevant.sum()), K)
                       if relevant.sum() > 0 else 0.0)
        results[K] = {
            'Recall': np.mean(recalls),
            'NDCG'  : np.mean(ndcgs),
            'mAP'   : np.mean(aps)
        }
    return results

def print_metrics(metrics, label):
    print(f'\n{"="*58}')
    print(f'  {label}')
    print(f'{"="*58}')
    print(f'{"K":<6} {"Recall@K":<12} {"NDCG@K":<12} {"mAP@K":<12}')
    print('-'*58)
    for K, v in metrics.items():
        print(f'@{K:<5} {v["Recall"]:.4f}      {v["NDCG"]:.4f}      {v["mAP"]:.4f}')
    print('='*58)

def run_retrieval(g_embs, q_embs, q_ids, label):
    set_seed(SEED)
    idx = faiss.IndexFlatIP(g_embs.shape[1])
    idx.add(g_embs.astype(np.float32))
    sims, idxs = idx.search(q_embs.astype(np.float32), 15)
    m = compute_metrics(q_ids, idxs, gallery_ids_arr)
    print_metrics(m, label)
    return m, sims, idxs

print('Metrics ready.')

In [ ]:

ALPHA_1 = 0.7   # 70% image, 30% text
ALPHA_2 = 0.5   # 50% image, 50% text

results = {}

for alpha in [ALPHA_1, ALPHA_2]:
    gf = fuse(gallery_embs_full, gallery_txt_full, alpha)
    qf = fuse(query_embs_full,   query_txt_full,   alpha)
    m, sims, idxs = run_retrieval(gf, qf, query_ids,
                                   f'B — Full Image | α={alpha}')
    results[f'full_{alpha}'] = (m, sims, idxs)

    gc = fuse(gallery_embs_crop, gallery_txt_crop, alpha)
    qc = fuse(query_embs_crop,   query_txt_crop,   alpha)
    m, sims, idxs = run_retrieval(gc, qc, query_ids,
                                   f'B — Cropped    | α={alpha}')
    results[f'crop_{alpha}'] = (m, sims, idxs)

In [ ]:

print('\n' + '='*65)
print(f'{"CONDITION B SUMMARY — Prompted BLIP-2":^65}')
print('='*65)
print(f'{"Condition":<35} {"R@5":<8} {"R@10":<8} {"R@15":<8} {"NDCG@5":<9} {"mAP@5"}')
print('-'*65)
for key, (m, _, __) in results.items():
    label = key.replace('_', ' α=').replace('full', 'Full').replace('crop', 'Crop')
    print(f'B — {label:<31} '
          f'{m[5]["Recall"]:.4f}   '
          f'{m[10]["Recall"]:.4f}   '
          f'{m[15]["Recall"]:.4f}   '
          f'{m[5]["NDCG"]:.4f}    '
          f'{m[5]["mAP"]:.4f}')
print('='*65)

In [ ]:

def show_retrievals(query_idx, idxs, sims, q_ids,
                    use_bbox_q=False, use_bbox_g=False,
                    top_k=5, title='', caption_q=None, caption_g=None):
    q_row = df_query.iloc[query_idx]
    q_id  = q_ids[query_idx]

    q_img = Image.open(q_row['image_path']).convert('RGB')
    if use_bbox_q:
        q_img = q_img.crop((q_row['x1'], q_row['y1'], q_row['x2'], q_row['y2']))

    ret_idxs = idxs[query_idx, :top_k]
    ret_sims = sims[query_idx, :top_k]
    ret_ids  = gallery_ids_arr[ret_idxs]

    fig, axes = plt.subplots(1, top_k + 1, figsize=(3*(top_k+1), 4))

    # Query
    axes[0].imshow(q_img)
    q_title = f'QUERY\n{"crop" if use_bbox_q else "full"}'
    if caption_q:
        q_title += f'\n"{caption_q[query_idx][:40]}..."'
    axes[0].set_title(q_title, fontsize=7, fontweight='bold')
    axes[0].axis('off')

    # Retrievals
    for i, (gidx, sim, gid) in enumerate(zip(ret_idxs, ret_sims, ret_ids)):
        g_row = df_gallery.iloc[gidx]
        g_img = Image.open(g_row['image_path']).convert('RGB')
        if use_bbox_g:
            g_img = g_img.crop((g_row['x1'], g_row['y1'], g_row['x2'], g_row['y2']))
        correct = (gid == q_id)

        g_title = f'{"✓" if correct else "✗"}  {sim:.3f}'
        if caption_g:
            g_title += f'\n"{caption_g[gidx][:35]}..."'
        axes[i+1].imshow(g_img)
        axes[i+1].set_title(g_title, fontsize=7,
                             color='green' if correct else 'red')
        axes[i+1].axis('off')
        for sp in axes[i+1].spines.values():
            sp.set_edgecolor('green' if correct else 'red')
            sp.set_linewidth(3)
            sp.set_visible(True)

    plt.suptitle(f'{title} | item_id: {q_id}', fontsize=9)
    plt.tight_layout()
    plt.show()

print('Visualization helper ready.')

In [ ]:

_, sims_f7, idxs_f7 = results['full_0.7']

good_f = [i for i, qid in enumerate(query_ids)
          if gallery_ids_arr[idxs_f7[i, 0]] == qid]
bad_f  = [i for i, qid in enumerate(query_ids)
          if gallery_ids_arr[idxs_f7[i, 0]] != qid]

print(f'Full α=0.7 — Good: {len(good_f)} | Bad: {len(bad_f)}')

print('\n=== B Full α=0.7 — GOOD RETRIEVALS (with captions) ===')
for idx in good_f[:4]:
    show_retrievals(idx, idxs_f7, sims_f7, query_ids,
                    use_bbox_q=False, use_bbox_g=False, top_k=5,
                    title='B-Full α=0.7 GOOD',
                    caption_q=query_captions_full,
                    caption_g=gallery_captions_full)

print('\n=== B Full α=0.7 — BAD RETRIEVALS (with captions) ===')
for idx in bad_f[:4]:
    show_retrievals(idx, idxs_f7, sims_f7, query_ids,
                    use_bbox_q=False, use_bbox_g=False, top_k=5,
                    title='B-Full α=0.7 BAD',
                    caption_q=query_captions_full,
                    caption_g=gallery_captions_full)

In [ ]:

_, sims_c7, idxs_c7 = results['crop_0.7']

good_c = [i for i, qid in enumerate(query_ids)
          if gallery_ids_arr[idxs_c7[i, 0]] == qid]
bad_c  = [i for i, qid in enumerate(query_ids)
          if gallery_ids_arr[idxs_c7[i, 0]] != qid]

print(f'Crop α=0.7 — Good: {len(good_c)} | Bad: {len(bad_c)}')

print('\n=== B Crop α=0.7 — GOOD RETRIEVALS (with captions) ===')
for idx in good_c[:4]:
    show_retrievals(idx, idxs_c7, sims_c7, query_ids,
                    use_bbox_q=True, use_bbox_g=True, top_k=5,
                    title='B-Crop α=0.7 GOOD',
                    caption_q=query_captions_crop,
                    caption_g=gallery_captions_crop)

print('\n=== B Crop α=0.7 — BAD RETRIEVALS (with captions) ===')
for idx in bad_c[:4]:
    show_retrievals(idx, idxs_c7, sims_c7, query_ids,
                    use_bbox_q=True, use_bbox_g=True, top_k=5,
                    title='B-Crop α=0.7 BAD',
                    caption_q=query_captions_crop,
                    caption_g=gallery_captions_crop)

In [ ]:

_, sims_f5, idxs_f5 = results['full_0.5']
_, sims_c5, idxs_c5 = results['crop_0.5']

good_f5 = [i for i, qid in enumerate(query_ids)
           if gallery_ids_arr[idxs_f5[i, 0]] == qid]
bad_f5  = [i for i, qid in enumerate(query_ids)
           if gallery_ids_arr[idxs_f5[i, 0]] != qid]

print('=== B Full α=0.5 — GOOD ===')
for idx in good_f5[:3]:
    show_retrievals(idx, idxs_f5, sims_f5, query_ids,
                    use_bbox_q=False, use_bbox_g=False, top_k=5,
                    title='B-Full α=0.5 GOOD',
                    caption_q=query_captions_full,
                    caption_g=gallery_captions_full)

print('\n=== B Full α=0.5 — BAD ===')
for idx in bad_f5[:3]:
    show_retrievals(idx, idxs_f5, sims_f5, query_ids,
                    use_bbox_q=False, use_bbox_g=False, top_k=5,
                    title='B-Full α=0.5 BAD',
                    caption_q=query_captions_full,
                    caption_g=gallery_captions_full)

In [ ]:

COMPARE_IDXS = [0, 5, 10, 50, 100, 200, 500]

for idx in COMPARE_IDXS:
    print(f'\n─── Query {idx} — Full image ───')
    show_retrievals(idx, idxs_f7, sims_f7, query_ids,
                    use_bbox_q=False, use_bbox_g=False, top_k=5,
                    title=f'B-Full α=0.7  idx={idx}',
                    caption_q=query_captions_full,
                    caption_g=gallery_captions_full)
    print(f'─── Query {idx} — Cropped ───')
    show_retrievals(idx, idxs_c7, sims_c7, query_ids,
                    use_bbox_q=True, use_bbox_g=True, top_k=5,
                    title=f'B-Crop α=0.7  idx={idx}',
                    caption_q=query_captions_crop,
                    caption_g=gallery_captions_crop)

In [ ]:

# Verify prompted captions are product-focused not scene-focused
print('CAPTION SPOT CHECK — first 8 gallery items\n')
print(f'{"idx":<5} {"FULL caption":<65} {"CROP caption"}')
print('-'*140)
for i in range(8):
    full = gallery_captions_full[i][:64]
    crop = gallery_captions_crop[i][:64]
    print(f'{i:<5} {full:<65} {crop}')

In [ ]:

zip_path = '/kaggle/working/condition_b_complete.zip'
with zipfile.ZipFile(zip_path, 'w') as z:
    for fname in os.listdir(SAVE_DIR):
        z.write(os.path.join(SAVE_DIR, fname), fname)

print('All files saved. Download link:')
FileLink(zip_path)